## LLAVA on SageMaker



In [1]:
%store -r

In [41]:
import boto3
import sagemaker
from sagemaker.utils import name_from_base
from sagemaker import image_uris
import jinja2
from pathlib import Path

In [42]:
llm_engine = "deepspeed"
# llm_engine = "fastertransformer"

In [43]:
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
sm_client = sagemaker_session.sagemaker_client
sm_runtime_client = sagemaker_session.sagemaker_runtime_client
s3_client = boto3.client('s3')
jinja_env = jinja2.Environment()
default_bucket = sagemaker_session.default_bucket()

In [44]:
!ls /home/ec2-user/SageMaker/vevor/categorymapping/vit_pairwise_model_emb/model.pth

/home/ec2-user/SageMaker/vevor/categorymapping/vit_pairwise_model_emb/model.pth


In [9]:
!aws s3 cp /home/ec2-user/SageMaker/vevor/categorymapping/vit_pairwise_model_emb/model.pth s3://sagemaker-us-west-2-726335585155/vevor-model/

upload: ../vit_pairwise_model_emb/model.pth to s3://sagemaker-us-west-2-726335585155/vevor-model/model.pth


In [45]:
framework_name = f"djl-{llm_engine}"
inference_image_uri = image_uris.retrieve(
    framework=framework_name, region=sagemaker_session.boto_session.region_name, version="0.23.0"
)

print(f"Inference container uri: {inference_image_uri}")

[12/06/24 03:46:24] INFO     Ignoring unnecessary instance type: None.                            ]8;id=204042;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=912811;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/image_uris.py#523\523]8;;\

Inference container uri: 763104351884.dkr.ecr.us-west-2.amazonaws.com/djl-inference:0.23.0-deepspeed0.9.5-cu118


In [46]:
s3_url = "s3://sagemaker-us-west-2-726335585155/vevor-model/model.pth"

In [48]:
%%writefile vit-src/serving.properties
engine=DeepSpeed
option.batch_size=16
#option.s3url=s3://sagemaker-us-west-2-726335585155/sagemaker-checkpoint-test/checkpoints-klook-0529-v2-10
option.model_id = s3://sagemaker-us-west-2-726335585155/vevor-model/

Overwriting vit-src/serving.properties


In [49]:
# we plug in the appropriate model location into our `serving.properties` file based on the region in which this notebook is running
!pygmentize vit-src/serving.properties | cat -n

     1	engine=DeepSpeed
     2	option.batch_size=16
     3	#option.s3url=s3://sagemaker-us-west-2-726335585155/sagemaker-checkpoint-test/checkpoints-klook-0529-v2-10
     4	option.model_id = s3://sagemaker-us-west-2-726335585155/vevor-model/


In [50]:
s3_target = f"s3://{sagemaker_session.default_bucket()}/vevor/"
print(s3_target)

s3://sagemaker-us-west-2-726335585155/vevor/


In [51]:
!rm vit-src.tar.gz
!tar zcvf vit-src.tar.gz vit-src --exclude ".ipynb_checkpoints" --exclude "__pycache__" --exclude ".ipynb"
!aws s3 cp vit-src.tar.gz {s3_target}

vit-src/
vit-src/model.py
vit-src/requirements.txt
vit-src/first_page_pic_infer.py
vit-src/run_llava_local.py
vit-src/serving.properties
upload: ./vit-src.tar.gz to s3://sagemaker-us-west-2-726335585155/vevor/vit-src.tar.gz


In [52]:
model_uri = f"{s3_target}vit-src.tar.gz"
print(model_uri)

s3://sagemaker-us-west-2-726335585155/vevor/vit-src.tar.gz


### 4.2 Create SageMaker endpoint

You need to specify the instance to use and endpoint names

In [ ]:
from sagemaker import Model, image_uris, serializers, deserializers

model = Model(image_uri=inference_image_uri, model_data=model_uri, role=role)

instance_type = "ml.g5.xlarge"
endpoint_name = sagemaker.utils.name_from_base("vevor-test")

model.deploy(initial_instance_count=1,
             instance_type=instance_type,
             endpoint_name=endpoint_name
            )

# our requests and responses will be in json format so we specify the serializer and the deserializer
predictor = sagemaker.Predictor(
    endpoint_name=endpoint_name,
    sagemaker_session=sagemaker_session,
    serializer=serializers.JSONSerializer(),
)

[12/06/24 03:47:02] INFO     Creating model with name: djl-inference-2024-12-06-03-47-02-833        ]8;id=783274;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=852513;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py#4025\4025]8;;\

[12/06/24 03:47:03] INFO     Creating endpoint-config with name vevor-test-2024-12-06-03-47-02-667  ]8;id=154919;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=242534;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py#5820\5820]8;;\

                    INFO     Creating endpoint with name vevor-test-2024-12-06-03-47-02-667         ]8;id=243527;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=247384;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py#4642\4642]8;;\

---

### invoke endpoint


In [61]:
%%time

data = {
    "input_image1" : 'https://raw.githubusercontent.com/haotian-liu/LLaVA/main/images/llava_logo.png',
    "input_image2" : 'https://raw.githubusercontent.com/haotian-liu/LLaVA/main/images/llava_logo.png',
    "text1" : 'test1',
    "text2" : 'test2'
}

# request
output = predictor.predict(data)
print(output)

b'{\n  "score":"0.15238057"\n}'
CPU times: user 3.43 ms, sys: 0 ns, total: 3.43 ms
Wall time: 176 ms


In [ ]:
%%time

data = {
    "input_image1" : 'https://raw.githubusercontent.com/haotian-liu/LLaVA/main/images/llava_logo.png',
    "input_image2" : 'https://raw.githubusercontent.com/haotian-liu/LLaVA/main/images/llava_logo.png',
    "text1" : 'test1',
    "text2" : 'test2'
}

# request
output = predictor.predict(data)
print(output)

In [62]:
%%time
import time
from tqdm import tqdm

# request
t0=time.time()
for i in tqdm(range(1000)):
    output = predictor.predict(data)
cost_time = time.time()-t0
print(cost_time)
print(output)

100%|██████████| 1000/1000 [01:58<00:00,  8.42it/s]

118.7125654220581
b'{\n  "score":"0.15238057"\n}'
CPU times: user 2.07 s, sys: 408 ms, total: 2.48 s
Wall time: 1min 58s


In [66]:
## multi process
# 压力测试, 多线程

import time
import random
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor


max_workers=64

def test_function(i):
    global data
    res = predictor.predict(data)

t0 = time.time()
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(test_function, i) for i in range(1000)]
    for future in tqdm(futures):
        future.result()
t1 = time.time()
dt = t1-t0


print (f"average time per 1000 image",dt )
print ("串行推理千张成本 - 1000 pic infer cost: ", dt/60/60*1.408)

100%|██████████| 1000/1000 [01:49<00:00,  9.13it/s]

average time per 1000 image 109.68954229354858
串行推理千张成本 - 1000 pic infer cost:  0.042900798763699


## delete endpoint

In [67]:
# delete sagemaker endpoint
predictor.delete_endpoint()